In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT=Path.cwd().parents[1]
sys.path.insert(0,str(ROOT))

import common as cfg
import helper_funcs as hf

CONFIG=cfg.load_config()
INPUT,OUTPUT,SRC=cfg.task_paths()
OUTPUT.mkdir(parents=True,exist_ok=True)
rng=cfg.initialize(CONFIG['seed'])


In [ ]:
MODEL='competing_risks'
print(hf.MODEL_LABELS[MODEL])
print(hf.hypothesis_text(MODEL))


In [ ]:
targets=pd.read_csv(INPUT/'b_prepare'/'empirical_targets.csv')
followup=pd.read_csv(INPUT/'b_prepare'/'followup_distribution.csv')


In [ ]:
draws,metrics,blocks=hf.prior_predictive(MODEL,CONFIG,followup,targets,rng)
check=hf.prior_predictive_check(metrics,targets,MODEL)


In [ ]:
draws.to_csv(OUTPUT/'prior_draws.csv',index=False)
metrics.to_csv(OUTPUT/'prior_predictive_metrics.csv',index=False)
blocks.to_csv(OUTPUT/'prior_block_distances.csv',index=False)
check.to_csv(OUTPUT/'prior_predictive_check.csv',index=False)


In [ ]:
plot=check.groupby('block',as_index=False).agg(median_error=('median_standardized_error','median'),coverage=('covered_90','mean'))
fig,axes=plt.subplots(1,2,figsize=(10,4.5))
sns.barplot(data=plot,y='block',x='median_error',ax=axes[0])
sns.barplot(data=plot,y='block',x='coverage',ax=axes[1])
axes[0].set(xlabel='median standardized error',ylabel='')
axes[1].set(xlabel='90% prior-predictive coverage',ylabel='',xlim=(0,1))
fig.suptitle(hf.MODEL_LABELS[MODEL])
fig.tight_layout()
fig.savefig(OUTPUT/'prior_predictive_check.png',dpi=220)
plt.close(fig)
plot
